# Notebook 02: Baseline U-Net Regression for Virtual Staining

### Overview
In this notebook, we train a 2D **U-Net** regression model to predict continuous fluorescent virus infection reporter intensity from label-free brightfield images.

**Key Learning Objectives**:
1. Define the 2D U-Net architecture with skip connections.
2. Train the model using Mean Absolute Error ($L_1$) loss.
3. Monitor training/validation loss curves.
4. Save model checkpoints and inspect virtual staining outputs.

In [ ]:
# ==========================================================
# 1. Google Colab Setup & Environment Setup
# ==========================================================
import sys
import os

if 'google.colab' in sys.modules:
    print("[+] Google Colab detected!")
    !git clone https://github.com/casus/GenAI_BIA_Course.git /content/GenAI_BIA_Course
    %cd /content/GenAI_BIA_Course/practical
    !pip install -r ../requirements.txt -q
    sys.path.append(os.path.abspath("."))
else:
    sys.path.append(os.path.abspath("."))

## 2. Load Dataset & Instantiate U-Net

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

from src.data import VIRVSDataset
from src.models import UNet
from src.generate_mock_virvs_data import create_dataset_directory
from src.utils import plot_virtual_staining_comparison

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[+] Using device: {device}")

# Ensure dataset exists
data_dir = "./data/mock_virvs"
create_dataset_directory(data_dir, num_train=30, num_val=10, image_size=(256, 256))

# U-Net uses range [0, 1]
train_dataset = VIRVSDataset(root_dir=data_dir, split="train", normalize_range=(0.0, 1.0))
val_dataset = VIRVSDataset(root_dir=data_dir, split="val", normalize_range=(0.0, 1.0))

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)

# Instantiate U-Net Model
model = UNet(in_channels=1, out_channels=1, features=[32, 64, 128, 256]).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.L1Loss()  # MAE Loss

## 3. Training Loop

In [ ]:
num_epochs = 15
train_losses, val_losses = [], []

print("[+] Training U-Net Virtual Staining Baseline...")
for epoch in range(1, num_epochs + 1):
    model.train()
    running_train_loss = 0.0
    
    for batch in train_loader:
        bf = batch["brightfield"].to(device)
        fluo = batch["fluorescence"].to(device)
        
        optimizer.zero_grad()
        pred = model(bf)
        loss = criterion(pred, fluo)
        loss.backward()
        optimizer.step()
        
        running_train_loss += loss.item() * bf.size(0)
        
    epoch_train_loss = running_train_loss / len(train_dataset)
    train_losses.append(epoch_train_loss)
    
    # Validation Step
    model.eval()
    running_val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            bf = batch["brightfield"].to(device)
            fluo = batch["fluorescence"].to(device)
            pred = model(bf)
            loss = criterion(pred, fluo)
            running_val_loss += loss.item() * bf.size(0)
            
    epoch_val_loss = running_val_loss / len(val_dataset)
    val_losses.append(epoch_val_loss)
    
    if epoch % 3 == 0 or epoch == num_epochs:
        print(f"Epoch [{epoch:02d}/{num_epochs:02d}] - Train L1: {epoch_train_loss:.4f} | Val L1: {epoch_val_loss:.4f}")

# Save trained U-Net checkpoint
torch.save(model.state_dict(), "./unet_virvs_baseline.pth")
print("[+] Model saved to unet_virvs_baseline.pth")

## 4. Plot Loss Curves & Visual Inspection

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(range(1, num_epochs + 1), train_losses, label="Train L1 Loss", fontweight="bold")
plt.plot(range(1, num_epochs + 1), val_losses, label="Val L1 Loss", linestyle="--", fontweight="bold")
plt.xlabel("Epoch")
plt.ylabel("MAE Loss")
plt.title("U-Net Virtual Staining Training Curve", fontweight="bold")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Visual comparison on validation sample
model.eval()
val_batch = next(iter(val_loader))
with torch.no_grad():
    bf_sample = val_batch["brightfield"][0:1].to(device)
    gt_sample = val_batch["fluorescence"][0:1].to(device)
    pred_sample = model(bf_sample)

plot_virtual_staining_comparison(
    bf_sample[0], gt_sample[0], pred_unet=pred_sample[0],
    title="U-Net Virtual Staining Prediction"
)